# 02 - Checks before modelling

Question: if two points are close together, do their temperatures move together? If so, a random train/test split would leak near-copies of the same weather into both sets.

In [ ]:
import pandas as pd
import numpy as np
# Line 1: what do the rows look like?
#Line 2: is the grid complete?
#Line 3: how much is missing

df = pd.read_parquet('../data/processed/dataset.parquet')
print(df.shape)
print(df[['pid', 'date', 'year', 'lat', 'lon', 'lst_target']].head(3))
print(df['pid'].nunique(), df['date'].nunique())
print(df['lst_target'].isna().mean().round(3))

(499968, 31)
   pid        date  year        lat      lon  lst_target
0    0  2003-06-02  2003  50.946084  0.11949         NaN
1    0  2003-06-10  2003  50.946084  0.11949       24.87
2    0  2003-06-18  2003  50.946084  0.11949       24.23
1984 252
0.179


In [ ]:
df['comp'] = df.groupby(['pid', 'year']).cumcount()
df['anom'] = df['lst_target'] - df.groupby(['pid', 'comp'])['lst_target'].transform('mean')

print(df[['pid', 'date', 'comp', 'lst_target', 'anom']].iloc[1:4])
print(df['anom'].describe().round(2))
wide = df.pivot(index='date', columns = 'pid', values = 'anom')
print(wide.shape)

   pid        date  comp  lst_target      anom
1    0  2003-06-10     1       24.87  1.621176
2    0  2003-06-18     2       24.23 -0.689412
3    0  2003-06-26     3       22.91 -2.152222
count    410417.00
mean         -0.00
std           3.78
min         -36.94
25%          -2.45
50%           0.04
75%           2.48
max          17.76
Name: anom, dtype: float64
(252, 1984)


In [4]:
corr = wide.corr(min_periods=100)
print(corr.shape)
print(corr.iloc[:3, :3].round(2))

(1984, 1984)
pid     0     1    2
pid                 
0    1.00  0.52  0.3
1    0.52  1.00  0.4
2    0.30  0.40  1.0


In [6]:
pts = df.drop_duplicates('pid').set_index('pid')[['lat', 'lon']].loc[corr.columns]
lat = np.radians(pts['lat'].to_numpy())
lon = np.radians(pts['lon'].to_numpy())

dlat = lat[:, None] - lat[None, :]
dlon = lon[:, None] - lon[None, :]

h = np.sin(dlat / 2)**2 + np.cos(lat[None, :]) * np.cos(lat[None, :]) * np.sin(dlon / 2)**2
dist = 2 * 6371 * np.arcsin(np.sqrt(h))
print(dist.shape)
print(dist[:3, :3].round(0))

(1984, 1984)
[[  0. 194. 727.]
 [197.   0. 574.]
 [737. 575.   0.]]


In [7]:
iu = np.triu_indices(len(dist), k=1)
pairs = pd.DataFrame({'km': dist[iu], 'r': corr.to_numpy()[iu]}).dropna()
print(len(iu[0]), len(pairs))

bins = [0, 10, 25, 50, 100, 200, 400, 800, 1300]
pairs['band'] = pd.cut(pairs['km'], bins)
print(pairs.groupby('band', observed=True)['r'].agg(['count', 'mean']).round(3))

1967136 1964868
              count   mean
band                      
(0, 10]        2387  0.787
(10, 25]      11669  0.729
(25, 50]      37973  0.695
(50, 100]    129709  0.656
(100, 200]   376019  0.607
(200, 400]   686905  0.535
(400, 800]   679765  0.399
(800, 1300]   40441  0.249


Finding: correlation of anomalies falls from about 0.79 (under 10 km) to 0.25 (over 800 km), and never reaches zero.
Why it matters: a random row-by-row split would put correlated, same-date rows in both train and test, giving optimistic scores.
Decision: train on 2003–2018, test on 2019–2023.
Limit: this split doesn't test predictions at unseen locations.